# Analisis estadistico — PROJECT_DESIGN seccion 7.B

Este notebook satisface PROJECT_DESIGN seccion 4 (que pide `notebooks/analysis.ipynb`) y seccion 6 Fase 7 (que pide ejecutar los 4 gates de la seccion 7.B en orden, solo lectura del .db, con veredicto AVANZA/FIN por gate).

**Toda la logica vive en `src/analysis/` para garantizar determinismo bit-exact.** Este notebook es un wrapper interactivo: llama las mismas funciones que `scripts/run_analysis.py`. Mismo input → mismo output_hash.

**Pre-registro:** los umbrales viven en `analysis_config.json` (commited en git). Modificarlos requiere bump de version + commit. Esto previene cherry-picking.

**Tecnicas de determinismo embebidas:**
1. Pre-registro JSON versionado
2. Hash SHA256 del input data
3. Modulo Python puro (esta notebook lo importa)
4. Synthetic fixtures con respuestas conocidas (`tests/test_analysis_fixtures.py`)
5. Bonferroni multiple-testing correction
6. K-fold cross-validation
7. Hash SHA256 del output

**Restriccion critica:** PROJECT_DESIGN linea 286: *'Antes de 1000 ventanas, cualquier numero es ruido.'* Si `n_after_filter < 1000`, todos los gates devuelven `INSUFICIENTE` y el veredicto global es `WAIT_FOR_MORE_DATA`.

## 1. Setup

In [ ]:
import sys
import json
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / 'analysis_config.json').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.analysis.config import load_config
from src.analysis.data import load_data
from src.analysis.gates import (
    gate1_d_vs_coin, gate2_d_vs_market, gate3_lead_lag,
    gate4a_chainlink_deribit, gate4b_costs, kfold_accuracy,
)
from src.analysis.stats import bonferroni_alpha, deterministic_train_holdout_split
from src.analysis.verdict import consolidate

print(f'ROOT: {ROOT}')

## 2. Cargar pre-registro (umbrales fijos, commited en git)

In [ ]:
config = load_config(ROOT / 'analysis_config.json')
print(f'Config version: {config.config_version}')
print(f'Frozen at:      {config.frozen_at_utc}')
print(f'\nGate 1 thresholds:')
print(f'  D_variant: {config.get("gate1", "D_variant")}')
print(f'  |D| threshold: {config.get("gate1", "D_threshold_abs")}')
print(f'  min accuracy to advance: {config.get("gate1", "min_accuracy_to_advance")}')
print(f'\nMin n para veredicto: {config.get("data", "min_n_for_verdict")}')

## 3. Load data + hash input

In [ ]:
data = load_data(config)
print(f'Total complete windows (sin filtro flujo): {data.n_total_complete}')
print(f'Despues de filtro min_n_trades_15m:        {data.n_after_filter}')
print(f'Input hash: {data.input_hash}')
if data.n_after_filter < int(config.get("data", "min_n_for_verdict")):
    print(f'\n⚠️  MUESTRA INSUFICIENTE: {data.n_after_filter} < 1000')
    print('   Los gates corren para validar codigo, pero veredicto sera WAIT_FOR_MORE_DATA.')

## 4. Train/holdout split (70/30)

Los gates se evaluan SOLO sobre el holdout, para evitar overfit.
El train se usa para K-fold CV (sanity check de varianza).

In [ ]:
train, holdout = deterministic_train_holdout_split(
    data.observations,
    ratio=float(config.get('split', 'train_holdout_ratio')),
    seed=int(config.get('split', 'random_seed')),
)
print(f'Train:   {len(train)}')
print(f'Holdout: {len(holdout)}')

## 5. Bonferroni correction

5 tests (Gate 1, 2, 3, 4a, 4b). α_corregido = 0.05 / 5 = 0.01

In [ ]:
alpha = float(config.get('gate1', 'alpha'))
n_tests = int(config.get('bonferroni', 'n_tests_total'))
alpha_corrected = bonferroni_alpha(alpha, n_tests)
min_n = int(config.get('data', 'min_n_for_verdict'))
print(f'Alpha uncorrected: {alpha}')
print(f'N tests:           {n_tests}')
print(f'Alpha corrected:   {alpha_corrected}')

## 6. GATE 1 — ¿D le gana a la moneda?

PROJECT_DESIGN 7.B: "De las ventanas donde |D| supero un umbral, sign(D) coincidio con outcome en mas del 50%?"
Metodo: accuracy direccional + test binomial.

In [ ]:
gate1 = gate1_d_vs_coin(holdout, config, alpha_corrected, min_n)
print(f'Gate 1 verdict: {gate1.verdict}')
print(f'  accuracy:  {gate1.primary_metric:.4f}')
print(f'  detail:\n{json.dumps(gate1.detail, indent=2)}')

## 7. GATE 2 — ¿D le gana al MERCADO indeciso?

PROJECT_DESIGN 7.B: filtrar `pm_up_price_open in [0.45, 0.55]`, medir accuracy en ESE subset.

In [ ]:
gate2 = gate2_d_vs_market(holdout, config, alpha_corrected, min_n)
print(f'Gate 2 verdict: {gate2.verdict}')
print(f'  accuracy:  {gate2.primary_metric:.4f}')
print(f'  detail:\n{json.dumps(gate2.detail, indent=2)}')

## 8. GATE 3 — ¿D se adelanta?

PROJECT_DESIGN 7.B: "analisis lead-lag". Mediana de `t_move` debe ser >= 30s para AVANZA.

In [ ]:
gate3 = gate3_lead_lag(holdout, data.trajectory, config, min_n)
print(f'Gate 3 verdict: {gate3.verdict}')
print(f'  median_t_move_seconds: {gate3.primary_metric}')
print(f'  detail:\n{json.dumps(gate3.detail, indent=2)}')

## 9. GATE 4a — ¿Chainlink-Deribit no se cancelan?

PROJECT_DESIGN 7.B: desfase deribit_open vs chainlink_close menor que movimiento tipico de ventana.

In [ ]:
gate4a = gate4a_chainlink_deribit(holdout, config, min_n)
print(f'Gate 4a verdict: {gate4a.verdict}')
print(f'  median ratio: {gate4a.primary_metric:.4f}')
print(f'  detail:\n{json.dumps(gate4a.detail, indent=2)}')

## 10. GATE 4b — ¿Sobrevive a costos reales?

PROJECT_DESIGN 7.B: backtest con spread + slippage + fees Polymarket (7% taker, 1¢ spread).

In [ ]:
gate4b = gate4b_costs(holdout, config, alpha_corrected, min_n)
print(f'Gate 4b verdict: {gate4b.verdict}')
print(f'  EV neto por apuesta: {gate4b.primary_metric:.6f}')
print(f'  detail:\n{json.dumps(gate4b.detail, indent=2)}')

## 11. K-fold sanity check sobre TRAIN

Solo valida estabilidad. Si mean_accuracy en train >> accuracy en holdout, hay overfit.

In [ ]:
kfold_result = kfold_accuracy(
    train,
    d_variant=config.get('gate1', 'D_variant'),
    d_threshold=float(config.get('gate1', 'D_threshold_abs')),
    n_splits=int(config.get('split', 'kfold_n_splits')),
    seed=int(config.get('split', 'random_seed')),
)
print(json.dumps(kfold_result, indent=2))

## 12. Veredicto consolidado + hash output

**Mismo input → mismo output_hash bit-exact.** Verifica corriendo el notebook 2 veces.

In [ ]:
summary = consolidate(
    gate_results=[gate1, gate2, gate3, gate4a, gate4b],
    config_version=config.config_version,
    input_hash=data.input_hash,
    n_total_complete=data.n_total_complete,
    n_after_filter=data.n_after_filter,
    min_n_for_verdict=min_n,
    extra={
        'n_train': len(train),
        'n_holdout': len(holdout),
        'kfold_train_accuracy': kfold_result,
        'bonferroni': {
            'alpha_uncorrected': alpha,
            'n_tests': n_tests,
            'alpha_corrected': alpha_corrected,
        },
    },
)
print('=' * 60)
print(f'PROJECT VERDICT: {summary["project_verdict"]}')
print('=' * 60)
print(f'config_version: {summary["config_version"]}')
print(f'input_hash:     {summary["input_hash"]}')
print(f'output_hash:    {summary["output_hash"]}')
print(f'n_after_filter: {summary["n_after_filter"]} / {min_n}')